# 🛠️ Enterprise HR Tool: Interactive Attrition Predictor
**Developer:** Sebastian Thurm  
**Application:** Real-time Risk Assessment & Bulk Reporting

---

## 💡 Overview
This dashboard is the operational front-end of the Attrition Risk Model. It is designed to provide HR Business Partners with immediate, actionable insights without requiring any coding knowledge.

### 🚀 Key Capabilities:
* **Individual Analysis:** Use interactive sliders to simulate "What-If" scenarios for specific employees.
* **Bulk Processing:** Upload a standard CSV export to generate a prioritized retention report for entire departments.
* **Ensemble Prediction:** Leverages a weighted average of Decision Tree logic and Logistic Regression for high-confidence scoring.

In [ ]:
# @title Importing and Libraries
!pip install pandas numpy scikit-learn imbalanced-learn ipywidgets -q

import pandas as pd
import numpy as np
import ipywidgets as widgets
import io
import base64
import warnings
from IPython.display import display, clear_output, HTML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report, recall_score, accuracy_score
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

# Code Toggle Script
display(HTML('''<script>
code_show=true;
function code_toggle() {
 if (code_show){ $('div.input').hide(); } else { $('div.input').show(); }
 code_show = !code_show
}
$( document ).ready(code_toggle);
</script>
<p><form action="javascript:code_toggle()"><input type="submit" value="Click here to Toggle Code On/Off"></form></p>'''))

# Data Loading
df = pd.read_csv('employee_data.csv')

# Strategic Feature Selection
features = ['OverTime', 'MonthlyIncome', 'TotalWorkingYears', 'JobLevel',
            'YearsAtCompany', 'MaritalStatus', 'DistanceFromHome', 'Age']

X = df[features].copy()
y = df['Attrition'].apply(lambda x: 1 if x == 'Yes' else 0)

# Encoding
X['OverTime'] = X['OverTime'].apply(lambda x: 1 if x == 'Yes' else 0)
X = pd.get_dummies(X, columns=['MaritalStatus'], drop_first=True)

# Fixed order for the model
cols = ['OverTime', 'MonthlyIncome', 'TotalWorkingYears', 'JobLevel', 'YearsAtCompany',
        'DistanceFromHome', 'Age', 'MaritalStatus_Married', 'MaritalStatus_Single']
X = X[cols]

print(f"✅ Data and Libraries Loaded successfully.")

In [ ]:
# @title ML Model and Training
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE

## 1. Split and Balance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_res, y_res = SMOTE(random_state=42).fit_resample(X_train, y_train)

# 2. Decision Tree: Tuned for Recall
dt_model = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
dt_model.fit(X_res, y_res)
dt_preds = dt_model.predict(X_test)

# 3. Logistic Regression: Tuned for Recall
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_res, y_res)

lr_threshold = 0.48
lr_probs = lr_model.predict_proba(X_test)[:, 1]
lr_preds = (lr_probs >= lr_threshold).astype(int)

# 4. Metrics & Stability Check
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_val_score(lr_model, X_res, y_res, cv=skf, scoring='recall')

print("--- MODEL COMPARISON: RECALL VS ACCURACY ---")
print(f"Decision Tree     | Recall: {recall_score(y_test, dt_preds):.2f} | Accuracy: {accuracy_score(y_test, dt_preds):.2f}")
print(f"Logistic Regress. | Recall: {recall_score(y_test, lr_preds):.2f} | Accuracy: {accuracy_score(y_test, lr_preds):.2f}")
print(f"CV Stability (Rec)| Mean: {cv_results.mean():.2f} (+/- {cv_results.std()*2:.2f})")

print("\n--- DETAILED CONFUSION MATRICES ---")
print("Decision Tree Matrix:")
print(confusion_matrix(y_test, dt_preds))
print("\nLogistic Regression Matrix (WINNER):")

print(confusion_matrix(y_test, lr_preds))

# 5. Set Winner of Training
model = lr_model

In [ ]:
# @title HR Attrition Tool

# 1. UI Styling & Help Section - Forced White Background
help_html = """
<div style="background-color: #ffffff !important; padding: 20px; border: 4px solid #d9534f; border-radius: 10px; font-family: Arial; color: #000000 !important;">
    <h3 style="margin-top:0; color: #d9534f !important;">📊 BULK UPLOAD SPECIFICATIONS</h3>
    <p style="color: #000000 !important;"><b>Required Headers (Copy exactly):</b></p>
    <div style="background-color: #eeeeee !important; padding: 10px; border: 1px solid #333; font-family: monospace; font-weight: bold; color: #b11d1d !important;">
        EmployeeID, OverTime, MonthlyIncome, TotalWorkingYears, JobLevel, YearsAtCompany, MaritalStatus, DistanceFromHome, Age
    </div>
</div>
"""

# 2. Define Widgets
style = {'description_width': 'initial'}
layout = widgets.Layout(width='450px', margin='8px 0px')
ot = widgets.Dropdown(options=[('Yes', 1), ('No', 0)], description='Frequent Overtime:', style=style, layout=layout)
income = widgets.IntSlider(value=5000, min=1000, max=20000, step=100, description='Monthly Salary ($):', style=style, layout=layout)
work_yrs = widgets.IntSlider(value=10, min=0, max=40, description='Total Career Years:', style=style, layout=layout)
level = widgets.SelectionSlider(options=[1, 2, 3, 4, 5], description='Job Level:', style=style, layout=layout)
co_yrs = widgets.IntSlider(value=5, min=0, max=40, description='Years at Company:', style=style, layout=layout)
marital = widgets.Dropdown(options=['Single', 'Married', 'Divorced'], description='Marital Status:', style=style, layout=layout)
dist = widgets.IntSlider(value=5, min=1, max=30, description='Distance (km):', style=style, layout=layout)
age = widgets.IntSlider(value=30, min=18, max=65, description='Employee Age:', style=style, layout=layout)
file_upload = widgets.FileUpload(accept='.csv', multiple=False, description="UPLOAD CSV", button_style='danger')
bulk_output = widgets.Output()

# 3. Processing Functions
def process_bulk_upload(change):
    with bulk_output:
        clear_output()
        if not file_upload.value: return
        input_file = list(file_upload.value.values())[0]
        bulk_df = pd.read_csv(io.BytesIO(input_file['content']))
        try:
            calc_df = bulk_df.copy()
            calc_df['OverTime'] = calc_df['OverTime'].map({'Yes': 1, 'No': 0})
            calc_df['MaritalStatus_Married'] = (calc_df['MaritalStatus'] == 'Married').astype(int)
            calc_df['MaritalStatus_Single'] = (calc_df['MaritalStatus'] == 'Single').astype(int)
            model_cols = ['OverTime', 'MonthlyIncome', 'TotalWorkingYears', 'JobLevel', 'YearsAtCompany', 'DistanceFromHome', 'Age', 'MaritalStatus_Married', 'MaritalStatus_Single']
            model_input = calc_df[model_cols]
            p_dt = dt_model.predict_proba(model_input)[:, 1]
            p_lr = lr_model.predict_proba(model_input)[:, 1]
            combined_probs = (p_dt + p_lr) / 2
            bulk_df['Risk_Score_%'] = (combined_probs * 100).round(1)
            bulk_df['Prediction'] = ['High Risk' if p >= 0.48 else 'Stable' for p in combined_probs]
            final_df = bulk_df[['EmployeeID', 'Prediction', 'Risk_Score_%'] + [c for c in bulk_df.columns if c not in ['EmployeeID', 'Prediction', 'Risk_Score_%']]]
            csv_str = final_df.to_csv(index=False)
            b64 = base64.b64encode(csv_str.encode()).decode()
            href = f'<a href="data:file/csv;base64,{b64}" download="Retention_Report.csv" style="color:white; background:#000; padding:12px; border-radius:5px; text-decoration:none; font-weight:bold; display:inline-block; border: 2px solid #d9534f;">📥 DOWNLOAD RESULTS</a>'
            display(HTML(f"<div style='margin-bottom:20px;'>{href}</div>"))
            # Forced colors for table preview
            display(HTML(f"<div style='background-color: white !important; color: black !important; padding: 10px;'>{final_df.head(5).to_html(index=False)}</div>"))
        except Exception as e:
            display(HTML(f"<div style='color:red;'>Error: {e}</div>"))

file_upload.observe(process_bulk_upload, names='value')

def ensemble_report(ot, income, work_yrs, level, co_yrs, marital, dist, age):
    m_married = 1 if marital == 'Married' else 0
    m_single = 1 if marital == 'Single' else 0
    input_df = pd.DataFrame([[ot, income, work_yrs, level, co_yrs, dist, age, m_married, m_single]],
                            columns=['OverTime', 'MonthlyIncome', 'TotalWorkingYears', 'JobLevel', 'YearsAtCompany', 'DistanceFromHome', 'Age', 'MaritalStatus_Married', 'MaritalStatus_Single'])
    p_dt = dt_model.predict_proba(input_df)[0][1]
    p_lr = lr_model.predict_proba(input_df)[0][1]
    prob = (p_dt + p_lr) / 2

    # FORCED DARK MODE COMPATIBILITY
    if prob >= 0.48:
        bg_color, status = "#d9534f", "🚨 RISK ALERT"
    else:
        bg_color, status = "#5cb85c", "✅ STABLE"

    display(HTML(f"""
        <div style='background-color:{bg_color} !important; padding:25px; border-radius:15px; border: 4px solid #000; width:450px; text-align:center; margin-top:20px;'>
            <h2 style='margin:0; color: #ffffff !important; font-size:32px; font-weight:bold;'>{status}</h2>
            <p style='font-size:24px; margin:15px 0; color: #ffffff !important;'><b>Risk Score: {prob*100:.1f}%</b></p>
            <div style='background-color:#ffffff; width:100%; height:20px; border:2px solid #000; border-radius:10px; overflow:hidden;'>
                <div style='background-color:#333; width:{prob*100}%; height:100%;'></div>
            </div>
        </div>
    """))

# 4. Display Session Interface
github_path = "Basti-T/ML_Portfolio/blob/main/01_Strategic-HR-Retention/01_Attrition_Risk_Analysis_Model.ipynb"
colab_link = f"https://colab.research.google.com/github/{github_path}"
badge_html = f"""
<div style="text-align:center; background:#333 !important; padding:15px; border-radius:5px; margin-bottom:20px;">
    <h2 style="color: white !important; margin-bottom: 10px;">Enterprise Attrition Tool</h2>
    <a href="{colab_link}" target="_blank">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
</div>
"""
display(HTML(badge_html))

out_ind = widgets.interactive_output(ensemble_report, {'ot':ot,'income':income,'work_yrs':work_yrs,'level':level,'co_yrs':co_yrs,'marital':marital,'dist':dist,'age':age})
tabs = widgets.Tab(children=[widgets.VBox([ot, income, work_yrs, level, co_yrs, marital, dist, age, out_ind]), widgets.VBox([widgets.HTML(help_html), file_upload, bulk_output])])
tabs.set_title(0, 'Individual Analysis')
tabs.set_title(1, 'Bulk CSV Upload')
display(tabs)